# 02 — Full train + HF adapter + HF Space Gradio deploy

Kaggle **GPU (T4)** notebook that:

1. Builds the grounded dataset (Phase 1)
2. Runs **full-epoch** Unsloth QLoRA SFT (Phase 2)
3. Runs a **side-by-side base vs fine-tuned** research-panel compare
4. Uploads the LoRA **adapter** to the Hugging Face Hub (model repo)
5. Deploys the **Gradio app to a Hugging Face Space** (permanent URL)
6. Optionally launches a temporary Gradio share from this runtime

| Flag | Default | Meaning |
|------|---------|---------|
| `RUN_TRAIN` | `True` | Actually train (needs T4) |
| `MAX_STEPS` | `None` | Full epoch from YAML |
| `RUN_SIDE_BY_SIDE` | `True` | Base vs adapter panel compare |
| `PUBLISH_HF` | `True` | Upload adapter model repo |
| `PUBLISH_SPACE` | `True` | Upload `spaces/ecra-demo` as HF Space |
| `LAUNCH_GRADIO` | `False` | Temporary Gradio share (blocks) |

**Secrets:** Kaggle → Add-ons → Secrets → `HF_TOKEN` (write). Never paste the token.

- Adapter model: `nuwanda94/llama32-3b-ecra-sft`
- Space URL after deploy: `https://huggingface.co/spaces/nuwanda94/earnings-call-research-assistant`

## 0. Knobs

In [ ]:
# --- User knobs ---
RUN_TRAIN = True
MAX_STEPS = None
MAX_SAMPLES = 50
DOWNLOAD_HF = True
USE_LLM_JUDGE = False
CONFIG_PATH = "configs/default.yaml"

RUN_SIDE_BY_SIDE = True
SIDE_BY_SIDE_LIMIT = 4

# Hugging Face — adapter (model repo)
PUBLISH_HF = True
HF_REPO_ID = "nuwanda94/llama32-3b-ecra-sft"
HF_PRIVATE = False

# Hugging Face — Gradio Space (permanent app)
PUBLISH_SPACE = True
SPACE_REPO_ID = "nuwanda94/earnings-call-research-assistant"
SPACE_PRIVATE = False
SPACE_DIR = "spaces/ecra-demo"

# Optional temporary Gradio from this runtime (not the Space)
LAUNCH_GRADIO = False
GRADIO_SHARE = True
GRADIO_SIDE_BY_SIDE = True

ADAPTER_DIR = "outputs/adapters/llama32-3b-ecra-sft"

print("RUN_TRAIN", RUN_TRAIN)
print("PUBLISH_HF", PUBLISH_HF, HF_REPO_ID)
print("PUBLISH_SPACE", PUBLISH_SPACE, SPACE_REPO_ID)
print("LAUNCH_GRADIO", LAUNCH_GRADIO)

## 1. Repo path + installs

In [ ]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    if not (repo / "src" / "earnings_call_research_assistant" / "inference.py").exists():
        %cd /kaggle/working
        !rm -rf earnings-call-research-assistant
        !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
        repo = work / "earnings-call-research-assistant"
    REPO = repo.resolve()
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)

import earnings_call_research_assistant as ecra
print("package:", ecra.__file__)

In [ ]:
if IN_KAGGLE:
    %pip install -q pyyaml huggingface_hub datasets
    if RUN_TRAIN or RUN_SIDE_BY_SIDE or LAUNCH_GRADIO:
        %pip install -q unsloth transformers accelerate bitsandbytes trl peft
    if LAUNCH_GRADIO:
        %pip install -q gradio

## 2. Hugging Face token (Kaggle secret)

Secret name: **`HF_TOKEN`** (write). Value is never printed.

In [ ]:
def _load_hf_token() -> bool:
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        return True
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
            if tok and tok.strip():
                os.environ["HF_TOKEN"] = tok.strip()
                return True
        except Exception as e:
            print("Kaggle secrets note:", type(e).__name__, str(e)[:120])
    return False

has_token = _load_hf_token()
print("HF token present:", has_token, "(value not shown)")
if (PUBLISH_HF or PUBLISH_SPACE) and not has_token:
    print("WARNING: publish flags on but no token.")

## 3. Phase 1 — data pipeline

In [ ]:
from earnings_call_research_assistant.data import (
    DATASET_VERSION, ChunkConfig, FilterConfig, GenerateConfig, SelectConfig,
    chunk_records, filter_pairs, generate_pairs, ingest_catalog, list_sources,
    select_and_split, write_chunks_jsonl, write_filter_report, write_jsonl,
    write_pairs_jsonl, write_splits,
)

for s in list_sources():
    print(f"  - {s.source_id}: {s.display_name}")

records = ingest_catalog(max_samples=MAX_SAMPLES, download=DOWNLOAD_HF)
write_jsonl(records, Path("data/raw/public_sample.jsonl"))
print("records:", len(records))

chunks = chunk_records(records, config=ChunkConfig(window_sentences=4, stride_sentences=2))
write_chunks_jsonl(chunks, Path("data/processed/chunks.jsonl"))
print("chunks:", len(chunks))

pairs = generate_pairs(
    chunks, config=GenerateConfig(max_qa_per_chunk=2, include_summary=True, use_llm=False)
)
write_pairs_jsonl(pairs, Path("data/processed/grounded_pairs.jsonl"))

kept, report = filter_pairs(
    pairs,
    config=FilterConfig(min_output_chars=40, near_dup_jaccard=0.88,
                        use_llm_judge=USE_LLM_JUDGE, min_judge_score=0.6),
)
write_pairs_jsonl(kept, Path("data/processed/filtered_pairs.jsonl"))
write_filter_report(report, Path("data/processed/filter_report.json"))
print("kept:", report.n_kept)

OUT_DIR = Path("data/processed") / DATASET_VERSION
sel_cfg = SelectConfig(
    target_min=1, target_max=6000, max_per_source=2500,
    diversity_jaccard_cap=0.72, seed=94, dataset_version=DATASET_VERSION,
)
splits, sel_report = select_and_split(kept, config=sel_cfg)
paths = write_splits(splits, OUT_DIR, report=sel_report, config=sel_cfg)
print("train:", sel_report.n_train, paths["train"])

## 4. SFT dry-run plan

In [ ]:
from earnings_call_research_assistant.training.sft import run_sft
import json

plan = run_sft(
    config_path=CONFIG_PATH, dataset_dir=OUT_DIR, dry_run=True,
    max_steps=MAX_STEPS, require_train=False,
)
print(f"model={plan.model_name} train={plan.n_train} adapter={plan.adapter_dir}")

## 5. Full QLoRA train

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

if not RUN_TRAIN:
    print("Skipped train.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable T4 GPU.")
    if plan.n_train < 1:
        raise RuntimeError("Empty train split.")
    train_plan = run_sft(
        config_path=CONFIG_PATH, dataset_dir=OUT_DIR, dry_run=False,
        max_steps=MAX_STEPS, require_train=True,
    )
    print("Train finished:", train_plan.adapter_dir)

adapter_path = Path(ADAPTER_DIR)
print("adapter exists:", adapter_path.exists())
if adapter_path.exists():
    print("files:", sorted(p.name for p in adapter_path.iterdir())[:15])

## 6. Side-by-side base vs fine-tuned

Writes `evals/reports/side_by_side_panel.jsonl`.

In [ ]:
import gc
from earnings_call_research_assistant.eval.panel import load_panel, DEFAULT_PANEL
from earnings_call_research_assistant.inference import InferenceConfig, InferenceHarness
import yaml

def _release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _load_cfg():
    with Path(CONFIG_PATH).open() as f:
        return InferenceConfig.from_mapping(yaml.safe_load(f))

def _generate_batch(harness, items):
    rows = []
    for item in items:
        text = item.user_text()
        reply = harness.generate(text)
        rows.append({"id": item.id, "ticker": item.ticker, "theme": item.theme,
                     "user_text": text, "reply": reply})
        print(f"  [{item.id}] -> {len(reply)} chars")
    return rows

compare_path = Path("evals/reports/side_by_side_panel.jsonl")
compare_path.parent.mkdir(parents=True, exist_ok=True)

if not RUN_SIDE_BY_SIDE:
    print("Skipped side-by-side.")
elif not torch.cuda.is_available() or not Path(ADAPTER_DIR).exists():
    print("Skipped side-by-side: need CUDA + adapter.")
else:
    panel = load_panel(DEFAULT_PANEL)[: max(1, int(SIDE_BY_SIDE_LIMIT))]
    cfg = _load_cfg()
    base_h = InferenceHarness.from_pretrained(cfg)
    base_rows = _generate_batch(base_h, panel)
    del base_h
    _release()
    try:
        tuned_h = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
    except Exception as e:
        print("adapter fallback:", e)
        tuned_h = InferenceHarness.from_pretrained(cfg)
        tuned_h.model.load_adapter(ADAPTER_DIR)
    tuned_rows = _generate_batch(tuned_h, panel)
    del tuned_h
    _release()
    by_id = {r["id"]: r for r in tuned_rows}
    with compare_path.open("w", encoding="utf-8") as f:
        for br in base_rows:
            tr = by_id.get(br["id"], {})
            row = {"id": br["id"], "ticker": br["ticker"], "theme": br["theme"],
                   "user_text": br["user_text"], "base_reply": br["reply"],
                   "adapter_reply": tr.get("reply", "")}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print("---", row["id"], "---")
            print("[BASE]", row["base_reply"][:300])
            print("[ADAPTER]", row["adapter_reply"][:300])
    print("Wrote", compare_path)

## 7. Publish adapter to HF Hub (model repo)

In [ ]:
from earnings_call_research_assistant.publish import publish_adapter

dry = publish_adapter(
    adapter_dir=ADAPTER_DIR, repo_id=HF_REPO_ID, private=HF_PRIVATE, dry_run=True,
)
print("adapter plan:", dry.adapter_exists, dry.looks_like_adapter, "token=", dry.token_present)

if not PUBLISH_HF:
    print("Skipped adapter upload.")
elif not dry.adapter_exists or not dry.looks_like_adapter:
    raise FileNotFoundError(ADAPTER_DIR)
elif not dry.token_present:
    raise RuntimeError("Set HF_TOKEN.")
else:
    live = publish_adapter(
        adapter_dir=ADAPTER_DIR, repo_id=HF_REPO_ID, private=HF_PRIVATE,
        commit_message="feat: upload ECRA QLoRA adapter from Kaggle", dry_run=False,
    )
    print("Adapter Hub:", live.hub_url)

## 8. Deploy Gradio app to Hugging Face **Space**

Uploads `spaces/ecra-demo` (`app.py` + `requirements.txt` + Space README) as a permanent Gradio Space.

After upload:
1. Open the Space URL
2. **Settings → Hardware → GPU (T4)** for live base vs adapter inference
3. Optional variables: `ADAPTER_REPO` (= `HF_REPO_ID`), `BASE_MODEL`

In [ ]:
from earnings_call_research_assistant.space_publish import publish_space

space_dry = publish_space(
    space_dir=SPACE_DIR, repo_id=SPACE_REPO_ID, private=SPACE_PRIVATE, dry_run=True,
)
print("space plan:", space_dry.has_required_files, "token=", space_dry.token_present)
print("files:", space_dry.files)

if not PUBLISH_SPACE:
    print("Skipped Space deploy.")
elif not space_dry.has_required_files:
    raise FileNotFoundError(SPACE_DIR)
elif not space_dry.token_present:
    raise RuntimeError("Set HF_TOKEN for Space upload.")
else:
    space_live = publish_space(
        space_dir=SPACE_DIR,
        repo_id=SPACE_REPO_ID,
        private=SPACE_PRIVATE,
        commit_message="feat: deploy ECRA Gradio Space from Kaggle notebook 02",
        dry_run=False,
    )
    print("Space URL:", space_live.hub_url)
    print("Next: Space Settings → Hardware → T4 for live weights.")
    print("Set variable ADAPTER_REPO=", HF_REPO_ID)

## 9. Optional temporary Gradio share (this runtime only)

Prefer the **Space** above for a permanent link. This cell is a short-lived `*.gradio.live` URL.

In [ ]:
if not LAUNCH_GRADIO:
    print("Skipped local Gradio. Use the HF Space URL from section 8.")
else:
    from earnings_call_research_assistant.demo import launch_demo
    adapter = ADAPTER_DIR if Path(ADAPTER_DIR).exists() else None
    launch_demo(
        load_model=True,
        config_path=CONFIG_PATH,
        adapter_dir=adapter,
        share=GRADIO_SHARE,
        server_name="0.0.0.0",
        side_by_side=GRADIO_SIDE_BY_SIDE,
    )

## Done

| Artifact | Where |
|----------|--------|
| Fine-tuned adapter (disk) | `outputs/adapters/llama32-3b-ecra-sft/` |
| Adapter on Hub | `https://huggingface.co/<HF_REPO_ID>` |
| Gradio **Space** (permanent) | `https://huggingface.co/spaces/<SPACE_REPO_ID>` |
| Side-by-side JSON | `evals/reports/side_by_side_panel.jsonl` |

Local re-deploy without Kaggle:

```bash
export HF_TOKEN=hf_xxx
python scripts/publish_adapter.py --run
python scripts/publish_space.py --run
```